In [9]:
import pandas as pd

# File paths
diagnoses_path = r"C:\Users\adity\Downloads\Machine learning\hackathon\archive\mimic-iii-clinical-database-demo-1.4\DIAGNOSES_ICD.csv"
noteevents_path = r"C:\Users\adity\Downloads\Machine learning\hackathon\archive\mimic-iii-clinical-database-demo-1.4\NOTEEVENTS.csv"
labevents_path = r"C:\Users\adity\Downloads\Machine learning\hackathon\archive\mimic-iii-clinical-database-demo-1.4\LABEVENTS.csv"
chartevents_path = r"C:\Users\adity\Downloads\Machine learning\hackathon\archive\mimic-iii-clinical-database-demo-1.4\CHARTEVENTS.csv"
icustays_path = r"C:\Users\adity\Downloads\Machine learning\hackathon\archive\mimic-iii-clinical-database-demo-1.4\ICUSTAYS.csv"
admissions_path = r"C:\Users\adity\Downloads\Machine learning\hackathon\archive\mimic-iii-clinical-database-demo-1.4\ADMISSIONS.csv"
patients_path = r"C:\Users\adity\Downloads\Machine learning\hackathon\archive\mimic-iii-clinical-database-demo-1.4\PATIENTS.csv"

# Load datasets
patients = pd.read_csv(patients_path)
admissions = pd.read_csv(admissions_path)
icustays = pd.read_csv(icustays_path)
chartevents = pd.read_csv(chartevents_path, usecols=['subject_id', 'hadm_id', 'itemid', 'charttime', 'value'], low_memory=False)
labevents = pd.read_csv(labevents_path, usecols=['subject_id', 'hadm_id', 'itemid', 'charttime', 'value'], low_memory=False)
noteevents = pd.read_csv(noteevents_path, usecols=['subject_id', 'hadm_id', 'charttime', 'text'], low_memory=False)
diagnoses = pd.read_csv(diagnoses_path, usecols=['subject_id', 'hadm_id', 'icd9_code'])
diagnoses.rename(columns={'icd9_code': 'icd_code'}, inplace=True)

# Preprocessing steps
# Handle missing values
patients.fillna(value={'gender': 'Unknown'}, inplace=True)
admissions.fillna(value={'admission_type': 'Unknown'}, inplace=True)
icustays.fillna(value={'los': 0}, inplace=True)

# Filter relevant features
patients = patients[['subject_id', 'gender', 'dob']]
admissions = admissions[['subject_id', 'hadm_id', 'admission_type', 'admittime']]
icustays = icustays[['subject_id', 'hadm_id', 'first_careunit', 'last_careunit', 'los']]
chartevents = chartevents.rename(columns={'value': 'vital_sign'})
labevents = labevents.rename(columns={'value': 'lab_value'})

# Convert 'vital_sign' to numeric, coercing non-numeric values to NaN
chartevents['vital_sign'] = pd.to_numeric(chartevents['vital_sign'], errors='coerce')

# Drop rows with NaN in 'vital_sign'
chartevents.dropna(subset=['vital_sign'], inplace=True)

# Merge datasets
# Merge patients and admissions
data = pd.merge(admissions, patients, on='subject_id', how='inner')

# Merge icustays
data = pd.merge(data, icustays, on=['subject_id', 'hadm_id'], how='inner')

# Merge CHARTEVENTS
chartevents_agg = chartevents.groupby(['subject_id', 'hadm_id']).agg({
    'vital_sign': ['mean', 'max', 'min']
}).reset_index()
# Flatten multi-level column names
chartevents_agg.columns = ['subject_id', 'hadm_id', 'vital_sign_mean', 'vital_sign_max', 'vital_sign_min']
data = pd.merge(data, chartevents_agg, on=['subject_id', 'hadm_id'], how='left')

# Merge LABEVENTS
labevents['lab_value'] = pd.to_numeric(labevents['lab_value'], errors='coerce')
labevents.dropna(subset=['lab_value'], inplace=True)
labevents_agg = labevents.groupby(['subject_id', 'hadm_id']).agg({
    'lab_value': ['mean', 'max', 'min']
}).reset_index()
labevents_agg.columns = ['subject_id', 'hadm_id', 'lab_value_mean', 'lab_value_max', 'lab_value_min']
data = pd.merge(data, labevents_agg, on=['subject_id', 'hadm_id'], how='left')

# Add sepsis flag from DIAGNOSES_ICD
diagnoses['sepsis_flag'] = diagnoses['icd_code'].apply(lambda x: 1 if str(x).startswith('995') else 0)
data = pd.merge(data, diagnoses[['subject_id', 'hadm_id', 'sepsis_flag']], on=['subject_id', 'hadm_id'], how='left')

# Merge NOTEEVENTS (if needed, process text data separately for NLP tasks)
noteevents_agg = noteevents.groupby(['subject_id', 'hadm_id']).agg({
    'text': lambda x: ' '.join(x)
}).reset_index()
data = pd.merge(data, noteevents_agg, on=['subject_id', 'hadm_id'], how='left')

# Fill NaN values with defaults
data.fillna(value=0, inplace=True)

# Save to CSV
output_path = r"C:\Users\adity\Downloads\Machine learning\hackathon\combined_dataset.csv"
data.to_csv(output_path, index=False)

print(f"Combined dataset saved to: {output_path}")


Combined dataset saved to: C:\Users\adity\Downloads\Machine learning\hackathon\combined_dataset.csv


C:\Users\adity\AppData\Local\Temp\ipykernel_1416\150695099.py:76: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data.fillna(value=0, inplace=True)


In [10]:
import pandas as pd

# Load dataset
file_path = r"C:\Users\adity\Downloads\Machine learning\hackathon\combined_dataset.csv"
data = pd.read_csv(file_path)

# Inspect dataset
print(data.head())


   subject_id  hadm_id admission_type            admittime gender  \
0       10006   142345      EMERGENCY  2164-10-23 21:09:00      F   
1       10006   142345      EMERGENCY  2164-10-23 21:09:00      F   
2       10006   142345      EMERGENCY  2164-10-23 21:09:00      F   
3       10006   142345      EMERGENCY  2164-10-23 21:09:00      F   
4       10006   142345      EMERGENCY  2164-10-23 21:09:00      F   

                   dob first_careunit last_careunit     los  vital_sign_mean  \
0  2094-03-05 00:00:00           MICU          MICU  1.6325        67.104367   
1  2094-03-05 00:00:00           MICU          MICU  1.6325        67.104367   
2  2094-03-05 00:00:00           MICU          MICU  1.6325        67.104367   
3  2094-03-05 00:00:00           MICU          MICU  1.6325        67.104367   
4  2094-03-05 00:00:00           MICU          MICU  1.6325        67.104367   

   vital_sign_max  vital_sign_min  lab_value_mean  lab_value_max  \
0           246.0             1.0   

In [11]:
print(data.isnull().sum())  # Check for remaining missing values


subject_id         0
hadm_id            0
admission_type     0
admittime          0
gender             0
dob                0
first_careunit     0
last_careunit      0
los                0
vital_sign_mean    0
vital_sign_max     0
vital_sign_min     0
lab_value_mean     0
lab_value_max      0
lab_value_min      0
sepsis_flag        0
text               0
dtype: int64


In [12]:
from sklearn.preprocessing import MinMaxScaler

# Select continuous columns for normalization
continuous_columns = ['vital_sign_mean', 'vital_sign_max', 'vital_sign_min', 
                      'lab_value_mean', 'lab_value_max', 'lab_value_min']

scaler = MinMaxScaler()
data[continuous_columns] = scaler.fit_transform(data[continuous_columns])

print(data[continuous_columns].head())


   vital_sign_mean  vital_sign_max  vital_sign_min  lab_value_mean  \
0         0.020861        0.000111             1.0        0.002469   
1         0.020861        0.000111             1.0        0.002469   
2         0.020861        0.000111             1.0        0.002469   
3         0.020861        0.000111             1.0        0.002469   
4         0.020861        0.000111             1.0        0.002469   

   lab_value_max  lab_value_min  
0       0.000645       0.983165  
1       0.000645       0.983165  
2       0.000645       0.983165  
3       0.000645       0.983165  
4       0.000645       0.983165  


In [13]:
# Example: Add rolling mean for vital_sign_mean over a window of 3 rows
data['rolling_vital_mean'] = data['vital_sign_mean'].rolling(window=3).mean()

# Example: Add rate of change for lab_value_mean
data['lab_value_mean_diff'] = data['lab_value_mean'].diff()

print(data[['vital_sign_mean', 'rolling_vital_mean', 'lab_value_mean_diff']].head())


   vital_sign_mean  rolling_vital_mean  lab_value_mean_diff
0         0.020861                 NaN                  NaN
1         0.020861                 NaN                  0.0
2         0.020861            0.020861                  0.0
3         0.020861            0.020861                  0.0
4         0.020861            0.020861                  0.0


In [2]:
import pandas as pd

# File paths
combined_dataset_path = r"C:\Users\adity\Downloads\Machine learning\hackathon\combined_dataset.csv"
chartevents_path = r"C:\Users\adity\Downloads\Machine learning\hackathon\archive\mimic-iii-clinical-database-demo-1.4\CHARTEVENTS.csv"
labevents_path = r"C:\Users\adity\Downloads\Machine learning\hackathon\archive\mimic-iii-clinical-database-demo-1.4\LABEVENTS.csv"

# Load combined dataset
combined_dataset = pd.read_csv(combined_dataset_path)

# Load CHARTEVENTS and LABEVENTS for extracting specific values
chartevents = pd.read_csv(chartevents_path, usecols=['subject_id', 'hadm_id', 'itemid', 'charttime', 'value'], low_memory=False)
labevents = pd.read_csv(labevents_path, usecols=['subject_id', 'hadm_id', 'itemid', 'charttime', 'value'], low_memory=False)

# Define itemid mappings for vital signs and lab values
vital_signs_mapping = {
    'heart_rate': [211, 220045],  # Item IDs for heart rate
    'temperature': [223761, 678],  # Item IDs for temperature
    'systolic_bp': [51, 442, 455, 6701, 220050],  # Item IDs for systolic blood pressure
    'diastolic_bp': [8368, 8441, 8555, 220051],  # Item IDs for diastolic blood pressure
    'respiratory_rate': [618, 615, 220210],  # Item IDs for respiratory rate
}

lab_tests_mapping = {
    'lactate': [50813],  # Item ID for lactate levels
}

# Extract vital signs
vital_signs = chartevents[chartevents['itemid'].isin(
    [item for sublist in vital_signs_mapping.values() for item in sublist]
)].copy()

# Map itemid to feature names
vital_signs['feature'] = vital_signs['itemid'].map({
    item: feature for feature, items in vital_signs_mapping.items() for item in items
})

# Pivot data to create separate columns for each feature
vital_signs_pivot = vital_signs.pivot_table(
    index=['subject_id', 'hadm_id'],
    columns='feature',
    values='value',
    aggfunc='first'
).reset_index()

# Convert vital signs to numeric
for column in vital_signs_mapping.keys():
    if column in vital_signs_pivot:
        vital_signs_pivot[column] = pd.to_numeric(vital_signs_pivot[column], errors='coerce')

# Extract lab tests
lab_tests = labevents[labevents['itemid'].isin(lab_tests_mapping['lactate'])].copy()

# Map itemid to feature name
lab_tests['feature'] = lab_tests['itemid'].map({50813: 'lactate'})

# Pivot lab tests data
lab_tests_pivot = lab_tests.pivot_table(
    index=['subject_id', 'hadm_id'],
    columns='feature',
    values='value',
    aggfunc='first'
).reset_index()

# Convert lactate values to numeric
lab_tests_pivot['lactate'] = pd.to_numeric(lab_tests_pivot['lactate'], errors='coerce')

# Merge extracted vital signs and lab tests with combined dataset
combined_dataset = combined_dataset.merge(vital_signs_pivot, on=['subject_id', 'hadm_id'], how='left')
combined_dataset = combined_dataset.merge(lab_tests_pivot, on=['subject_id', 'hadm_id'], how='left')

# Save updated dataset
output_path = r"C:\Users\adity\Downloads\Machine learning\hackathon\updated_combined_dataset.csv"
combined_dataset.to_csv(output_path, index=False)

print(f"Updated combined dataset saved to: {output_path}")


Updated combined dataset saved to: C:\Users\adity\Downloads\Machine learning\hackathon\updated_combined_dataset.csv


In [3]:
pip install spacy

Note: you may need to restart the kernel to use updated packages.
